# LangGraph G9 — Reliability and limits
Real tools fail, real models loop. CampusAI v8 needs three things every production agent has:

- **Retries** for transient failures, declared on the node (`retry_policy`), not written into every tool.
- **Errors as data**: a tool that raises should become a `ToolMessage` the model can read, not a crash.
- **Limits**: a `recursion_limit` on the run so a confused model cannot loop forever and burn credit.

```text
tool raises -> RetryPolicy retries the node -> still failing? -> ToolNode(handle_tool_errors=True) returns an error message -> model reacts
model loops -> recursion_limit -> GraphRecursionError -> your code stops the run safely
```

A note on retries: retry **reads** freely. Never blindly retry a **write**: if the network dropped
after the registration went through, a retry registers twice. Side-effecting tools need an
idempotency key so the server can recognise a repeat. That is distributed-systems engineering,
and agents inherit all of it.

### Step 1 — A flaky tool, a retry policy on the node, and errors as messages

> **Why LangGraph has *RetryPolicy* on a node**
>
> Every tool could wrap itself in a retry loop, but then each author decides differently and side effects get retried by accident. Declaring the policy on the node gives one rule for all tools in it, keeps tool code plain, and lets you say precisely which errors are transient.

In [ ]:
from langgraph.types import RetryPolicy                    # LangGraph: declarative retries for a node

TIMETABLE_ATTEMPTS = {"count": 0}                          # ours: counts calls to the flaky tool

@tool
def get_timetable(student_id: str) -> str:
    """Get a student's weekly timetable. (Flaky: the first call times out.)"""
    TIMETABLE_ATTEMPTS["count"] += 1
    if TIMETABLE_ATTEMPTS["count"] == 1:
        raise TimeoutError("timetable service timed out")
    return json.dumps({"student_id": student_id, "monday": "CS201 09:00", "tuesday": "MA110 11:00"})

def agent_tt(state: ChatState):                            # ours
    reply = model.bind_tools(KNOWLEDGE_TOOLS + [get_timetable]).invoke([SystemMessage(CAMPUS_PERSONA)] + state["messages"])   # LangChain
    return {"messages": [reply]}

g = StateGraph(ChatState)
g.add_node("agent", agent_tt)
g.add_node("tools", ToolNode(KNOWLEDGE_TOOLS + [get_timetable], handle_tool_errors=False),   # LangGraph: let exceptions escape so the retry policy sees them
           retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.1, retry_on=TimeoutError))   # LangGraph: retry the node up to 3 times.
# retry_on matters: the default predicate retries network-style errors only and deliberately skips
# ValueError, TypeError, OSError and friends, because retrying a bug never helps. Name what is transient.
g.add_edge(START, "agent"); g.add_conditional_edges("agent", tools_condition); g.add_edge("tools", "agent")
resilient = g.compile()

out = resilient.invoke({"messages": [HumanMessage("Show the timetable for S002.")]})
print("attempts:", TIMETABLE_ATTEMPTS["count"], "| answer:", text_of(out["messages"][-1])[:100])

# Without retries: ask ToolNode to turn the exception into an error ToolMessage the model can read.
TIMETABLE_ATTEMPTS["count"] = 0
p = StateGraph(ChatState)
p.add_node("tools", ToolNode(KNOWLEDGE_TOOLS + [get_timetable], handle_tool_errors=True))   # LangGraph: ANY exception -> error ToolMessage (the default only converts bad-argument errors)
p.add_edge(START, "tools"); p.add_edge("tools", END)
ai = AIMessage(content="", tool_calls=[{"name": "get_timetable", "args": {"student_id": "S002"}, "id": "call_1"}])   # LangChain: a hand-made request
print("error as data:", text_of(p.compile().invoke({"messages": [ai]})["messages"][-1])[:90])

> **What just happened**
>
> First run: `agent` -> `tools`, where get_timetable raised TimeoutError. Because the node has a RetryPolicy that names TimeoutError, LangGraph ran the tools node again; the second call succeeded (attempts: 2) and the loop continued as normal. Second demo: without a retry policy, `handle_tool_errors=True` turned the same exception into an error ToolMessage, the kind of text a model can read and react to.

### Step 2 — Bounding the loop with `recursion_limit`

Every node execution counts as one step. The default limit is 25; the toy graph below loops
forever, so a small limit stops it with `GraphRecursionError`. Pass the same option to any agent
run as insurance against a model that keeps calling tools.

> **Why LangGraph has a *recursion limit***
>
> A graph with a loop has no natural end; a model that keeps asking for tools can run forever and spend real money. The limit counts node executions (steps) rather than Python recursion, and it lives in the run config so the same graph can be given different budgets.

In [ ]:
from langgraph.errors import GraphRecursionError           # LangGraph

def spin(state: Counter):                                  # ours: a node that never wants to stop
    return {"value": state["value"] + 1, "trace": state["trace"] + "."}

g = StateGraph(Counter); g.add_node("spin", spin); g.add_edge(START, "spin")
g.add_conditional_edges("spin", lambda s: "spin", ["spin"])   # LangGraph: always loop back
forever = g.compile()
try:
    forever.invoke({"value": 0, "trace": ""}, config={"recursion_limit": 6})   # LangGraph: at most 6 steps
except GraphRecursionError as exc:
    print("stopped safely:", str(exc)[:70], "...")

# The same insurance on a real agent run: 8 steps = at most 4 model calls + 4 tool rounds.
out = resilient.invoke({"messages": [HumanMessage("What is the retake rule?")]}, config={"recursion_limit": 8})
print("agent within limit ->", text_of(out["messages"][-1])[:100])

> **What just happened**
>
> The toy graph's conditional edge always returned `"spin"`, so nothing would ever reach END. `recursion_limit: 6` counted node executions and raised GraphRecursionError at the sixth; the except block turned that into a clean message. The agent run with limit 8 finished normally in far fewer steps, which is the point: the limit is insurance, not a target.

### Recap

- **The problem we started with:** a tool timeout crashed the run, and nothing stopped a looping model.
- **What we added:** `RetryPolicy` with an explicit `retry_on`, ToolNode's `handle_tool_errors`, and `recursion_limit` on the run.
- **What you saw in the output:** the flaky tool succeeded on attempt 2; the endless graph stopped at step 6 with a clean exception.
- **Carry forward:** G10 runs independent checks at the same time instead of one after another.